# RF Diffusion Implementation
https://github.com/RosettaCommons/RFdiffusion

## Setup

### Accept Terms of Service

In [1]:
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/msys2

accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
accepted Terms of Service for https://repo.anaconda.com/pkgs/msys2


### Create Conda environment

In [2]:
!conda env create -f ../Tools/RFdiffusion/env/SE3nv.yml

2 channel Terms of Service accepted
Channels:
 - defaults
 - conda-forge
 - pytorch
 - dglteam
 - nvidia
Platform: linux-64
Solving environment: done

cudatoolkit-11.1.1   | 929.6 MB  |                                       |   0% 
dgl-cuda11.1-0.9.1po | 223.5 MB  |                                       |   0% 

mkl-2021.4.0         | 142.6 MB  |                                       |   0% 


pytorch-1.9.1        | 45.2 MB   |                                       |   0% 



scipy-1.10.1         | 23.1 MB   |                                       |   0% 




python-3.9.24        | 23.1 MB   |                                       |   0% 





torchvision-0.15.2   | 9.8 MB    |                                       |   0% 






numpy-base-1.24.3    | 6.9 MB    |                                       |   0% 







torchaudio-0.9.1     | 4.4 MB    |                                       |   0% 








intel-openmp-2021.4. | 4.2 MB    |                                       |   0% 





Please select the SE3nv Conda Environment from the Kernel Selector in VS Code

In [1]:
# Note that these commands are listed but cannot be executed in the notebook directly.
# Use the kernel selector to activate conda. The next block of code will point to the folder directly
!conda activate SE3nv


CondaError: Run 'conda init' before 'conda activate'



### Use `pip` to set up packages

*`cd` command coes not work directly in VS code*

In [2]:
import os
cur_dir = os.getcwd()
os.chdir('../Tools/RFdiffusion/env/SE3Transformer')

%pip install --no-cache-dir -r requirements.txt
!python setup.py install # Depricated

os.chdir(cur_dir)

  Cloning https://github.com/NVIDIA/dllogger to /tmp/pip-install-39teg88e/dllogger_9a758c601e9d44cb9a084238aad2ab0f
  Running command git clone --filter=blob:none --quiet https://github.com/NVIDIA/dllogger /tmp/pip-install-39teg88e/dllogger_9a758c601e9d44cb9a084238aad2ab0f
  Resolved https://github.com/NVIDIA/dllogger to commit 0478734ff7be75adde8d160e04872664d1c62e5f
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 10.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 11.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 11.5 MB/s  0:00:00
  DEPRECATION: Building 'dllogger' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build

### Install RFdiffusion

Does not want to run in VS Code. Can do setup with Conda terminal

In [3]:
os.chdir('../Tools/RFdiffusion')
%pip install -e . # install the rfdiffusion module from the root of the repository

os.chdir(cur_dir)

Obtaining file:///home/ryangustafson/Documents/GitHub/PhD-Research/Tools/RFdiffusion
  Preparing metadata (setup.py) ... done
  DEPRECATION: Legacy editable install of rfdiffusion==1.1.0 from file:///home/ryangustafson/Documents/GitHub/PhD-Research/Tools/RFdiffusion (setup.py develop) is deprecated. pip 25.3 will enforce this behaviour change. A possible replacement is to add a pyproject.toml or enable --use-pep517, and use setuptools >= 64. If the resulting installation is not behaving as expected, try using --config-settings editable_mode=compat. Please consult the setuptools documentation for more information. Discussion can be found at https://github.com/pypa/pip/issues/11457
  Running setup.py develop for rfdiffusion
Note: you may need to restart the kernel to use updated packages.


## Use RFDiffusion

Once the environment is set up, just use the Kernel picker to use the environment

In [1]:
import os, time, subprocess

original_directory = os.getcwd()

data_path = "../Data"
rf_diff_path = "../Tools/RFdiffusion/scripts"

pdb_path = os.path.join(data_path, "TIMP3_vs_ADAM17_X_ray.pdb")
output_prefix = "../Local/rfdiffusion_output/design"


loop_insertion_site = 30
contig_string = "B1-30/6-6/B36-121" # @param {type:"string"}
num_sequences_to_generate = 50 # @param {type:"integer"}

print(original_directory)

/home/ryangustafson/Documents/GitHub/PhD-Research/Generation


In [5]:
os.environ["HYDRA_FULL_ERROR"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [6]:
# --- Run RFdiffusion ---
if not pdb_path:
    print("Cannot run RFdiffusion without a scaffold PDB file.")
    raise Exception("No PDB File")

print("Preparing to run RFdiffusion...")

# Construct the command for RFdiffusion
run_command = [
    "python",
    os.path.join(rf_diff_path.replace('../', ''), "run_inference.py"),
    f"inference.output_prefix={output_prefix.replace('../', '')}",
    f"inference.input_pdb={pdb_path.replace('../', '')}",
    f"contigmap.contigs=[{contig_string}]",
    #"denoiser.num_steps=50",
    f"inference.num_designs={num_sequences_to_generate}",
]

print("Running RFdiffusion to generate novel loops and structures...")
print(" ".join(run_command))

# Run the command and stream output live
os.chdir("..") 
st = time.time()
result = subprocess.run(run_command, capture_output=True, text=True)
end = time.time()
os.chdir(original_directory)

print("--- STDOUT ---")
print(result.stdout)

print("--- STDERR ---")
print(result.stderr)

print(f"RFdiffusion finished in {(end-st)/60:.2f} minutes.")

Preparing to run RFdiffusion...
Running RFdiffusion to generate novel loops and structures...
python Tools/RFdiffusion/scripts/run_inference.py inference.output_prefix=Local/rfdiffusion_output/design inference.input_pdb=Data/TIMP3_vs_ADAM17_X_ray.pdb contigmap.contigs=[B1-30/6-6/B36-121] inference.num_designs=50
--- STDOUT ---
[2025-11-02 21:24:15,873][__main__][INFO] - Found GPU with device_name NVIDIA GeForce RTX 5070 Ti. Will run RFdiffusion on NVIDIA GeForce RTX 5070 Ti
Reading models from /home/ryangustafson/Documents/GitHub/PhD-Research/Tools/RFdiffusion/rfdiffusion/inference/../../models
[2025-11-02 21:24:15,874][rfdiffusion.inference.model_runners][INFO] - Reading checkpoint from /home/ryangustafson/Documents/GitHub/PhD-Research/Tools/RFdiffusion/rfdiffusion/inference/../../models/Base_ckpt.pt
This is inf_conf.ckpt_path
/home/ryangustafson/Documents/GitHub/PhD-Research/Tools/RFdiffusion/rfdiffusion/inference/../../models/Base_ckpt.pt
Assembling -model, -diffuser and -preprocess

In [ ]:
# --- Parse PDB Results to Extract Sequences ---
print("parsing generated PDBs to extract sequences...")

# 3-letter to 1-letter amino acid code map
aa_map = {'CYS': 'C', 'ASP': 'D', 'SER': 'S', 'GLN': 'Q', 'LYS': 'K',
        'ILE': 'I', 'PRO': 'P', 'THR': 'T', 'PHE': 'F', 'ASN': 'N',
        'GLY': 'G', 'HIS': 'H', 'LEU': 'L', 'ARG': 'R', 'TRP': 'W',
        'ALA': 'A', 'VAL': 'V', 'GLU': 'E', 'TYR': 'Y', 'MET': 'M'}

generated_sequences = set()
chain_id_to_extract = contig_string.split('/')[0][0] # Get chain from contig
start_res, end_res = map(int, contig_string.split('/')[1].split('-'))
loop_length_range = range(start_res, end_res + 1)

for i in range(num_sequences_to_generate):
    pdb_file = f"{output_prefix}_{i}.pdb"
    if os.path.exists(pdb_file):
        current_sequence = []
        with open(pdb_file, 'r') as f:
            for line in f:
                if line.startswith('ATOM') and line[21] == chain_id_to_extract:
                    res_name = line[17:20]
                    if res_name in aa_map:
                        current_sequence.append(aa_map[res_name])

        # The generated loop is appended at the end of the chain in RFdiffusion outputs
        loop_seq = "".join(current_sequence[-len(current_sequence) + loop_insertion_site :])
        if len(loop_seq) in loop_length_range:
                generated_sequences.add(loop_seq)

In [ ]:
# Final cleanup
original_sequences = set(df['sequence'])
unique_new_sequences = list(generated_sequences - original_sequences)

print(f"\nExtracted {len(unique_new_sequences)} unique, novel loop sequences.")
print("Here are a few examples:")
for seq in unique_new_sequences[:5]:
    print(f"   - {seq}")